# UCSD-PTGBM Final Factorial Statistics + Publication Figures

This notebook is the **single downstream analysis notebook** for the completed 12-run experiment.

It consumes:

- Seed 42: baseline, ADC-only, RSI-only, full 6-channel
- Seed 43: baseline, ADC-only, RSI-only, full 6-channel
- Seed 44: baseline, ADC-only, RSI-only, full 6-channel

It performs:

- fail-closed discovery and provenance validation;
- subject-level seed aggregation;
- the pre-specified 2×2 RSI × ADC factorial analysis;
- class-specific NETC / SNFH / ET / RC analyses;
- multiplicity correction;
- seed-robustness analysis;
- publication-ready main figures;
- supplementary figures;
- manuscript-ready result tables;
- a complete analysis manifest;
- a ZIP containing all final statistical outputs and figures.

## Scientific hierarchy

**Primary estimand**

\[
\Delta_{\mathrm{RSI}}
=
\frac{1}{2}
\left[
(D_{\mathrm{RSI}}-D_{\mathrm{Base}})
+
(D_{\mathrm{Full}}-D_{\mathrm{ADC}})
\right]
\]

**Key secondary estimands**

\[
\Delta_{\mathrm{ADC}}
=
\frac{1}{2}
\left[
(D_{\mathrm{ADC}}-D_{\mathrm{Base}})
+
(D_{\mathrm{Full}}-D_{\mathrm{RSI}})
\right]
\]

\[
I_{\mathrm{RSI}\times\mathrm{ADC}}
=
D_{\mathrm{Full}}
-D_{\mathrm{RSI}}
-D_{\mathrm{ADC}}
+D_{\mathrm{Base}}
\]

The statistical unit is the **subject**. Seed 42/43/44 are repeated model fits and are **not treated as independent patients**.

The current `internal_test` split has already been inspected during model development and is therefore labeled a **development holdout**, not an untouched external test set.

## 1. Imports, frozen configuration, and output folders

In [ ]:
from pathlib import Path
import re
import json
import hashlib
import shutil
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

# ---------------------------
# Frozen analysis definition
# ---------------------------

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
    Path.cwd(),
]

PRIMARY_SPLIT = "internal_test"
ANALYSIS_STATUS = "development_holdout"

ARMS = ["baseline", "adc_only", "rsi_only", "full_6ch"]
ARM_DISPLAY = {
    "baseline": "Baseline",
    "adc_only": "ADC-only",
    "rsi_only": "RSI-only",
    "full_6ch": "Full",
}

SEEDS = [42, 43, 44]

EXPECTED_CHANNEL_INDICES = {
    "baseline": [0, 1, 2, 3],
    "adc_only": [0, 1, 2, 3, 5],
    "rsi_only": [0, 1, 2, 3, 4],
    "full_6ch": [0, 1, 2, 3, 4, 5],
}

CHANNEL_NAMES = {
    0: "T1",
    1: "T1ce",
    2: "T2",
    3: "FLAIR",
    4: "RSI Cellularity",
    5: "ADC",
}

LABEL_NAMES = {
    1: "NETC",
    2: "SNFH",
    3: "ET",
    4: "RC",
}

LABEL_LONG_NAMES = {
    1: "Non-enhancing Tumor Core",
    2: "Surrounding Non-enhancing FLAIR Hyperintensity",
    3: "Enhancing Tissue",
    4: "Resection Cavity",
}

METRICS = {
    "foreground_mean_dice": "Foreground mean Dice",
    "dice_class_1": "NETC Dice",
    "dice_class_2": "SNFH Dice",
    "dice_class_3": "ET Dice",
    "dice_class_4": "RC Dice",
}

N_BOOT = 20000
BOOTSTRAP_SEED = 20261011

REQUIRE_ALL_12_RUNS = True
REQUIRE_RUN_METADATA = True

# Output root.
OUTPUT_DIR = Path("/kaggle/working/ucsd_ptgbm_final_factorial_analysis")
if not Path("/kaggle/working").exists():
    OUTPUT_DIR = Path.cwd() / "ucsd_ptgbm_final_factorial_analysis"

FIG_MAIN = OUTPUT_DIR / "figures_main"
FIG_SUPP = OUTPUT_DIR / "figures_supplementary"
TABLE_DIR = OUTPUT_DIR / "tables"
DATA_DIR = OUTPUT_DIR / "derived_data"

for d in [OUTPUT_DIR, FIG_MAIN, FIG_SUPP, TABLE_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Analysis status:", ANALYSIS_STATUS)
print("Output directory:", OUTPUT_DIR.resolve())

## 2. Discover the 12 subject-level metric exports automatically

Expected filename pattern:

`internal_test_<arm>_seed<seed>_split-<hash>_subject_metrics.csv`

The 12 runs may live in deeply nested directories. This cell searches recursively, so the current Kaggle dataset structure does not need to be flattened or renamed.

In [ ]:
FILE_RE = re.compile(
    rf"^{re.escape(PRIMARY_SPLIT)}_"
    r"(baseline|adc_only|rsi_only|full_6ch)_seed(\d+)_split-([0-9a-fA-F]+)_subject_metrics\.csv$"
)

def sha256_file(path, chunk_size=4 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

found = []
seen_paths = set()

for root in SEARCH_ROOTS:
    root = Path(root)
    if not root.exists():
        continue

    print("Searching:", root)

    try:
        for p in root.rglob("*_subject_metrics.csv"):
            try:
                pkey = str(p.resolve())
            except Exception:
                pkey = str(p)

            if pkey in seen_paths:
                continue
            seen_paths.add(pkey)

            match = FILE_RE.match(p.name)
            if match is None:
                continue

            arm, seed, split_hash = match.groups()
            seed = int(seed)

            if arm not in ARMS or seed not in SEEDS:
                continue

            found.append({
                "arm": arm,
                "seed": seed,
                "split_hash_from_filename": split_hash,
                "path": p,
                "file_sha256": sha256_file(p),
            })
    except Exception as exc:
        print(f"Warning while searching {root}: {exc}")

if not found:
    raise RuntimeError(
        "No matching internal-test subject-metric CSV files were found under the search roots."
    )

found_df = pd.DataFrame(found)
display(found_df[["arm", "seed", "split_hash_from_filename", "path", "file_sha256"]])

# Allow duplicate mounted copies only if they are byte-identical.
registry = {}
conflicts = []

for (arm, seed), grp in found_df.groupby(["arm", "seed"], sort=True):
    hashes = grp["file_sha256"].unique()

    if len(hashes) > 1:
        conflicts.append({
            "arm": arm,
            "seed": int(seed),
            "paths": [str(x) for x in grp["path"]],
            "hashes": list(hashes),
        })
    else:
        registry[(arm, int(seed))] = Path(grp.iloc[0]["path"])

if conflicts:
    display(pd.DataFrame(conflicts))
    raise RuntimeError(
        "Conflicting duplicate files exist for at least one arm/seed. "
        "Remove stale duplicate results before analysis."
    )

expected = {(arm, seed) for arm in ARMS for seed in SEEDS}
missing = sorted(expected - set(registry))

print("\nUnique discovered run keys:")
for key in sorted(registry):
    print(" ", key)

print("\nMissing:", missing)

if REQUIRE_ALL_12_RUNS and missing:
    raise RuntimeError(f"Missing completed runs: {missing}")

if REQUIRE_ALL_12_RUNS and len(registry) != 12:
    raise RuntimeError(f"Expected 12 unique runs, found {len(registry)}")

print("\nPASS: all 12 required subject-metric exports were found.")

## 3. Validate run metadata and prove the 12 runs are comparable

In [ ]:
def find_nearby_metadata(metric_path):
    candidates = [metric_path.parent / "run_metadata.json"]

    for parent in list(metric_path.parents)[:5]:
        candidates.append(parent / "run_metadata.json")

    for p in candidates:
        if p.exists():
            return p

    return None


metadata = {}
metadata_rows = []

for key, metric_path in sorted(registry.items()):
    arm, seed = key
    mpath = find_nearby_metadata(metric_path)

    if mpath is None:
        metadata[key] = None
        metadata_rows.append({
            "arm": arm,
            "seed": seed,
            "metadata_status": "MISSING",
            "metric_path": str(metric_path),
            "metadata_path": None,
        })
        continue

    with mpath.open("r") as f:
        obj = json.load(f)

    metadata[key] = obj
    rc = obj.get("run_config", {})

    metadata_rows.append({
        "arm": arm,
        "seed": seed,
        "metadata_status": "OK",
        "metric_path": str(metric_path),
        "metadata_path": str(mpath),
        "config_arm": rc.get("experiment_mode"),
        "config_seed": rc.get("run_seed"),
        "split_sha256": rc.get("split_sha256"),
        "dataset_id_sha256": rc.get("dataset_id_sha256"),
        "label_content_sha256": rc.get("label_content_sha256"),
        "structural_content_sha256": rc.get("structural_content_sha256"),
        "advanced_channel_content_sha256": rc.get("advanced_channel_content_sha256"),
        "pretrained_sha256": rc.get("pretrained_sha256"),
        "in_channels": rc.get("in_channels"),
        "selected_channel_indices": rc.get("selected_channel_indices"),
    })

metadata_df = pd.DataFrame(metadata_rows)
display(metadata_df)

if REQUIRE_RUN_METADATA and (metadata_df["metadata_status"] != "OK").any():
    raise RuntimeError(
        "At least one run_metadata.json is missing. "
        "Keep each run's metadata together with the exported metrics."
    )

# Validate arm, seed, and input-channel contract.
for (arm, seed), obj in metadata.items():
    if obj is None:
        continue

    rc = obj.get("run_config", {})

    if rc.get("experiment_mode") != arm:
        raise RuntimeError(f"{arm}/seed{seed}: experiment_mode mismatch")

    if int(rc.get("run_seed")) != seed:
        raise RuntimeError(f"{arm}/seed{seed}: run_seed mismatch")

    observed_indices = list(rc.get("selected_channel_indices", []))
    expected_indices = EXPECTED_CHANNEL_INDICES[arm]

    if observed_indices != expected_indices:
        raise RuntimeError(
            f"{arm}/seed{seed}: selected channels mismatch. "
            f"Expected {expected_indices}, got {observed_indices}."
        )

    if int(rc.get("in_channels")) != len(expected_indices):
        raise RuntimeError(f"{arm}/seed{seed}: in_channels mismatch")


FIXED_KEYS = [
    "split_seed",
    "split_sha256",
    "dataset_id_sha256",
    "label_content_sha256",
    "structural_content_sha256",
    "advanced_channel_content_sha256",
    "num_classes",
    "architecture",
    "feature_size",
    "roi_size",
    "crop_ratios",
    "batch_size",
    "num_samples_per_volume",
    "gradient_accumulation_steps",
    "learning_rate",
    "weight_decay",
    "val_every",
    "early_stop_patience",
    "max_epochs",
    "class_weights_sha256",
    "pretrained_sha256",
    "sliding_window_overlap",
    "sliding_window_mode",
    "absence_rule",
    "checkpoint_selection",
]

ref_key = sorted(registry)[0]
ref_config = metadata[ref_key]["run_config"]

mismatches = []

for key in sorted(registry):
    rc = metadata[key]["run_config"]

    for field in FIXED_KEYS:
        if rc.get(field) != ref_config.get(field):
            mismatches.append({
                "run": f"{key[0]}_seed{key[1]}",
                "field": field,
                "reference_value": ref_config.get(field),
                "run_value": rc.get(field),
            })

if mismatches:
    display(pd.DataFrame(mismatches))
    raise RuntimeError(
        "At least one controlled pipeline/provenance field differs across runs."
    )

print("PASS: controlled metadata are identical across the 12 runs.")
print("Split SHA256:            ", ref_config.get("split_sha256"))
print("Dataset-ID SHA256:       ", ref_config.get("dataset_id_sha256"))
print("Label-content SHA256:    ", ref_config.get("label_content_sha256"))
print("Structural-content SHA256:", ref_config.get("structural_content_sha256"))
print("Advanced-content SHA256: ", ref_config.get("advanced_channel_content_sha256"))
print("Pretrained SHA256:       ", ref_config.get("pretrained_sha256"))

## 4. Load all subject-level metrics and verify exact pairing

In [ ]:
REQUIRED_COLUMNS = [
    "subject_id",
    "foreground_mean_dice",
    "dice_class_1",
    "dice_class_2",
    "dice_class_3",
    "dice_class_4",
]

data = {}
load_rows = []

for (arm, seed), path in sorted(registry.items()):
    df = pd.read_csv(path)

    missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_cols:
        raise RuntimeError(f"{path.name}: missing columns {missing_cols}")

    if df["subject_id"].duplicated().any():
        dups = df.loc[df["subject_id"].duplicated(), "subject_id"].tolist()
        raise RuntimeError(f"{path.name}: duplicate subject IDs {dups[:10]}")

    df = df.set_index("subject_id").sort_index()
    data[(arm, seed)] = df

    load_rows.append({
        "arm": arm,
        "seed": seed,
        "n_subjects": int(len(df)),
        "n_missing_foreground": int(df["foreground_mean_dice"].isna().sum()),
        "path": str(path),
    })

load_df = pd.DataFrame(load_rows)
display(load_df)

ref_subjects = list(data[sorted(data)[0]].index)

bad_subject_sets = []

for key, df in data.items():
    if list(df.index) != ref_subjects:
        bad_subject_sets.append({
            "run": f"{key[0]}_seed{key[1]}",
            "n_subjects": len(df),
            "only_in_reference": sorted(set(ref_subjects) - set(df.index))[:20],
            "only_in_run": sorted(set(df.index) - set(ref_subjects))[:20],
        })

if bad_subject_sets:
    display(pd.DataFrame(bad_subject_sets))
    raise RuntimeError("Subject pairing differs across the 12 runs.")

print(f"PASS: identical paired subject set across all 12 runs (n={len(ref_subjects)}).")

# Long-format export for complete transparency.
long_rows = []

for (arm, seed), df in sorted(data.items()):
    temp = df[REQUIRED_COLUMNS[1:]].copy().reset_index()
    temp["arm"] = arm
    temp["seed"] = seed
    temp["analysis_status"] = ANALYSIS_STATUS
    long_rows.append(temp)

long_df = pd.concat(long_rows, ignore_index=True)
long_df.to_csv(DATA_DIR / "subject_seed_arm_long.csv", index=False)

display(long_df.head())

## 5. Statistical helper functions

In [ ]:
def holm_correction(pvals):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)

    if n == 0:
        return np.array([], dtype=float)

    order = np.argsort(pvals)
    adjusted = np.empty(n, dtype=float)
    running_max = 0.0

    for rank, idx in enumerate(order):
        candidate = pvals[idx] * (n - rank)
        running_max = max(running_max, candidate)
        adjusted[idx] = min(running_max, 1.0)

    return adjusted


def matched_rank_biserial(diffs):
    d = np.asarray(diffs, dtype=float)
    d = d[np.isfinite(d)]
    d = d[d != 0]

    if len(d) == 0:
        return 0.0

    ranks = stats.rankdata(np.abs(d))
    positive = ranks[d > 0].sum()
    negative = ranks[d < 0].sum()

    return float((positive - negative) / ranks.sum())


def bootstrap_ci(values, statistic=np.mean, n_boot=N_BOOT, alpha=0.05, seed=BOOTSTRAP_SEED):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) < 2:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot, dtype=float)

    for i in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        boots[i] = statistic(sample)

    lo, hi = np.quantile(boots, [alpha / 2, 1 - alpha / 2])
    return float(lo), float(hi)


def paired_signflip_p(values, n_perm=100000, seed=BOOTSTRAP_SEED + 1):
    """
    Paired sign-flip permutation sensitivity test against zero.
    Uses the absolute mean effect as the test statistic.
    """
    d = np.asarray(values, dtype=float)
    d = d[np.isfinite(d)]

    if len(d) < 2:
        return np.nan

    observed = abs(float(np.mean(d)))
    rng = np.random.default_rng(seed)

    extreme = 0
    chunk = 10000

    for start in range(0, n_perm, chunk):
        m = min(chunk, n_perm - start)
        signs = rng.choice([-1.0, 1.0], size=(m, len(d)))
        stats_perm = np.abs((signs * d).mean(axis=1))
        extreme += int(np.sum(stats_perm >= observed))

    return float((extreme + 1) / (n_perm + 1))


def wilcoxon_against_zero(values):
    d = np.asarray(values, dtype=float)
    d = d[np.isfinite(d)]

    if len(d) < 3:
        return np.nan, np.nan

    if np.allclose(d, 0):
        return 0.0, 1.0

    try:
        stat, p = stats.wilcoxon(
            d,
            zero_method="wilcox",
            alternative="two-sided",
            method="auto",
        )
        return float(stat), float(p)
    except ValueError:
        return np.nan, np.nan


def summarize_contrast(values, estimand, metric, analysis_family):
    d = pd.Series(values, dtype=float).dropna().to_numpy()

    mean_lo, mean_hi = bootstrap_ci(d, np.mean)
    median_lo, median_hi = bootstrap_ci(d, np.median)
    w_stat, w_p = wilcoxon_against_zero(d)
    perm_p = paired_signflip_p(d)

    return {
        "analysis_status": ANALYSIS_STATUS,
        "analysis_family": analysis_family,
        "metric": metric,
        "estimand": estimand,
        "n_subjects": int(len(d)),
        "mean_effect": float(np.mean(d)),
        "mean_effect_ci95_lo": mean_lo,
        "mean_effect_ci95_hi": mean_hi,
        "median_effect": float(np.median(d)),
        "median_effect_ci95_lo": median_lo,
        "median_effect_ci95_hi": median_hi,
        "wilcoxon_stat": w_stat,
        "wilcoxon_p": w_p,
        "signflip_permutation_p": perm_p,
        "rank_biserial": matched_rank_biserial(d),
        "n_positive": int((d > 0).sum()),
        "n_negative": int((d < 0).sum()),
        "n_zero": int((d == 0).sum()),
        "positive_fraction": float((d > 0).mean()),
    }


def seed_averaged_arm_table(metric, strict_complete_12=True):
    per_arm = {}

    for arm in ARMS:
        frame = pd.concat(
            [data[(arm, seed)][metric].rename(f"seed{seed}") for seed in SEEDS],
            axis=1,
        )
        # Require all 3 seeds in the arm to be finite when strict_complete_12=True
        per_arm[arm] = frame.mean(axis=1, skipna=not strict_complete_12)

    out = pd.DataFrame(per_arm)
    if strict_complete_12:
        # Require all 4 arms (all 12 runs) to be finite for the subject to contribute
        incomplete_mask = out.isna().any(axis=1)
        out.loc[incomplete_mask, :] = np.nan

    out.index.name = "subject_id"
    return out


def factorial_contrasts(arm_df):
    c = pd.DataFrame(index=arm_df.index)

    c["rsi_main_effect"] = 0.5 * (
        (arm_df["rsi_only"] - arm_df["baseline"])
        + (arm_df["full_6ch"] - arm_df["adc_only"])
    )

    c["adc_main_effect"] = 0.5 * (
        (arm_df["adc_only"] - arm_df["baseline"])
        + (arm_df["full_6ch"] - arm_df["rsi_only"])
    )

    c["rsi_x_adc_interaction"] = (
        arm_df["full_6ch"]
        - arm_df["rsi_only"]
        - arm_df["adc_only"]
        + arm_df["baseline"]
    )

    c["rsi_effect_adc_absent"] = arm_df["rsi_only"] - arm_df["baseline"]
    c["rsi_effect_adc_present"] = arm_df["full_6ch"] - arm_df["adc_only"]
    c["adc_effect_rsi_absent"] = arm_df["adc_only"] - arm_df["baseline"]
    c["adc_effect_rsi_present"] = arm_df["full_6ch"] - arm_df["rsi_only"]
    c["full_minus_baseline"] = arm_df["full_6ch"] - arm_df["baseline"]

    return c


def save_figure_all_formats(fig, stem, main=True, dpi=300):
    folder = FIG_MAIN if main else FIG_SUPP

    for ext in ["png", "pdf", "svg"]:
        path = folder / f"{stem}.{ext}"
        if ext == "png":
            fig.savefig(path, dpi=dpi, bbox_inches="tight")
        else:
            fig.savefig(path, bbox_inches="tight")

    return folder / f"{stem}.png"


print("Statistical and figure helpers ready.")

## 6. Seed-average each arm and calculate subject-level factorial contrasts

In [ ]:
global_arm_scores = seed_averaged_arm_table("foreground_mean_dice")
global_contrasts = factorial_contrasts(global_arm_scores)

global_arm_scores.to_csv(DATA_DIR / "subject_seed_averaged_arm_scores_global.csv")
global_contrasts.to_csv(DATA_DIR / "subject_factorial_contrasts_global.csv")

print("Seed-averaged arm scores:")
display(global_arm_scores.head())

print("\nSubject-level factorial contrasts:")
display(global_contrasts.head())

## 7. PRIMARY analysis — RSI main effect

In [ ]:
primary_df = pd.DataFrame([
    summarize_contrast(
        global_contrasts["rsi_main_effect"],
        estimand="RSI main effect",
        metric="Foreground mean Dice",
        analysis_family="PRIMARY",
    )
])

display(primary_df)
primary_df.to_csv(TABLE_DIR / "Table_primary_rsi_main_effect.csv", index=False)

## 8. Key secondary global analyses — ADC main effect and RSI × ADC interaction

In [ ]:
secondary_df = pd.DataFrame([
    summarize_contrast(
        global_contrasts["adc_main_effect"],
        estimand="ADC main effect",
        metric="Foreground mean Dice",
        analysis_family="KEY_SECONDARY_GLOBAL",
    ),
    summarize_contrast(
        global_contrasts["rsi_x_adc_interaction"],
        estimand="RSI × ADC interaction",
        metric="Foreground mean Dice",
        analysis_family="KEY_SECONDARY_GLOBAL",
    ),
])

secondary_df["holm_p"] = holm_correction(
    secondary_df["wilcoxon_p"].fillna(1.0).to_numpy()
)
secondary_df["holm_significant_0.05"] = secondary_df["holm_p"] < 0.05

display(secondary_df)
secondary_df.to_csv(TABLE_DIR / "Table_key_secondary_global.csv", index=False)

## 9. Class-specific RSI effects — mechanistic secondary analysis

In [ ]:
class_rsi_rows = []
class_arm_scores = {}
class_contrasts = {}

for class_id in range(1, 5):
    metric = f"dice_class_{class_id}"
    arm_scores = seed_averaged_arm_table(metric)
    contrasts = factorial_contrasts(arm_scores)

    class_arm_scores[class_id] = arm_scores
    class_contrasts[class_id] = contrasts

    row = summarize_contrast(
        contrasts["rsi_main_effect"],
        estimand="RSI main effect",
        metric=f"{LABEL_NAMES[class_id]} Dice",
        analysis_family="CLASS_SPECIFIC_RSI",
    )

    row["class_id"] = class_id
    row["class_name"] = LABEL_NAMES[class_id]
    row["class_long_name"] = LABEL_LONG_NAMES[class_id]
    class_rsi_rows.append(row)

class_rsi_df = pd.DataFrame(class_rsi_rows)

class_rsi_df["holm_p"] = holm_correction(
    class_rsi_df["wilcoxon_p"].fillna(1.0).to_numpy()
)
class_rsi_df["holm_significant_0.05"] = class_rsi_df["holm_p"] < 0.05

display(class_rsi_df)
class_rsi_df.to_csv(TABLE_DIR / "Table_class_specific_rsi_holm4.csv", index=False)

## 10. Exploratory class-specific ADC effects and RSI × ADC interactions

In [ ]:
exploratory_rows = []

for class_id in range(1, 5):
    contrasts = class_contrasts[class_id]

    for col, label in [
        ("adc_main_effect", "ADC main effect"),
        ("rsi_x_adc_interaction", "RSI × ADC interaction"),
    ]:
        row = summarize_contrast(
            contrasts[col],
            estimand=label,
            metric=f"{LABEL_NAMES[class_id]} Dice",
            analysis_family="EXPLORATORY_CLASS_ADC_INTERACTION",
        )

        row["class_id"] = class_id
        row["class_name"] = LABEL_NAMES[class_id]
        exploratory_rows.append(row)

exploratory_class_df = pd.DataFrame(exploratory_rows)

exploratory_class_df["holm_p"] = holm_correction(
    exploratory_class_df["wilcoxon_p"].fillna(1.0).to_numpy()
)
exploratory_class_df["holm_significant_0.05"] = exploratory_class_df["holm_p"] < 0.05

display(exploratory_class_df)
exploratory_class_df.to_csv(
    TABLE_DIR / "Table_exploratory_class_adc_interaction_holm8.csv",
    index=False,
)

## 11. Simple effects and conventional Full-vs-Baseline comparison

In [ ]:
simple_specs = [
    ("rsi_effect_adc_absent", "RSI effect when ADC absent"),
    ("rsi_effect_adc_present", "RSI effect when ADC present"),
    ("adc_effect_rsi_absent", "ADC effect when RSI absent"),
    ("adc_effect_rsi_present", "ADC effect when RSI present"),
]

simple_rows = []

for col, label in simple_specs:
    simple_rows.append(
        summarize_contrast(
            global_contrasts[col],
            estimand=label,
            metric="Foreground mean Dice",
            analysis_family="SIMPLE_EFFECTS_EXPLORATORY",
        )
    )

simple_df = pd.DataFrame(simple_rows)
simple_df["holm_p"] = holm_correction(simple_df["wilcoxon_p"].fillna(1.0).to_numpy())
simple_df["holm_significant_0.05"] = simple_df["holm_p"] < 0.05

full_vs_baseline_df = pd.DataFrame([
    summarize_contrast(
        global_contrasts["full_minus_baseline"],
        estimand="Full 6-channel − baseline",
        metric="Foreground mean Dice",
        analysis_family="DESCRIPTIVE_CONVENTIONAL_COMPARISON",
    )
])

display(simple_df)
display(full_vs_baseline_df)

simple_df.to_csv(TABLE_DIR / "Table_simple_effects_holm4.csv", index=False)
full_vs_baseline_df.to_csv(
    TABLE_DIR / "Table_descriptive_full_vs_baseline.csv",
    index=False,
)

## 12. Seed-by-seed robustness analysis

In [ ]:
def arm_table_for_seed(metric, seed):
    return pd.DataFrame({
        arm: data[(arm, seed)][metric]
        for arm in ARMS
    })

seed_rows = []
seed_subject_rows = []

for seed in SEEDS:
    arm_scores = arm_table_for_seed("foreground_mean_dice", seed)
    contrasts = factorial_contrasts(arm_scores)

    subject_temp = contrasts.copy().reset_index()
    subject_temp["seed"] = seed
    seed_subject_rows.append(subject_temp)

    for col, label in [
        ("rsi_main_effect", "RSI main effect"),
        ("adc_main_effect", "ADC main effect"),
        ("rsi_x_adc_interaction", "RSI × ADC interaction"),
        ("full_minus_baseline", "Full 6-channel − baseline"),
    ]:
        row = summarize_contrast(
            contrasts[col],
            estimand=label,
            metric="Foreground mean Dice",
            analysis_family="SEED_ROBUSTNESS",
        )
        row["seed"] = seed
        seed_rows.append(row)

seed_robustness_df = pd.DataFrame(seed_rows)
seed_subject_df = pd.concat(seed_subject_rows, ignore_index=True)

display(seed_robustness_df)

seed_robustness_df.to_csv(TABLE_DIR / "Table_seed_robustness.csv", index=False)
seed_subject_df.to_csv(DATA_DIR / "seed_subject_factorial_contrasts.csv", index=False)

## 13. Descriptive arm-level summary

In [ ]:
arm_rows = []

for arm in ARMS:
    values = global_arm_scores[arm].dropna().to_numpy()
    mean_lo, mean_hi = bootstrap_ci(values, np.mean)

    arm_rows.append({
        "analysis_status": ANALYSIS_STATUS,
        "arm": arm,
        "arm_display": ARM_DISPLAY[arm],
        "n_subjects": int(len(values)),
        "mean_foreground_dice": float(np.mean(values)),
        "mean_ci95_lo": mean_lo,
        "mean_ci95_hi": mean_hi,
        "median_foreground_dice": float(np.median(values)),
        "sd_across_subjects": float(np.std(values, ddof=1)),
    })

arm_descriptive_df = pd.DataFrame(arm_rows)

display(arm_descriptive_df)
arm_descriptive_df.to_csv(TABLE_DIR / "Table_arm_descriptive.csv", index=False)

# Publication figures

The following figures are generated automatically from the same locked analysis objects used for the statistical tables.

All main and supplementary figures are exported in:

- **PNG** (300 dpi)
- **PDF**
- **SVG**

No statistical figure is redrawn manually outside this notebook.

## 14. Figure 1 — Study design and 2×2 factorial framework

In [ ]:
def subject_from_timepoint_id(tid):
    m = re.fullmatch(r"(.+)_([0-9]+)", str(tid))
    return m.group(1) if m else str(tid)


def discover_frozen_split():
    candidates = []

    for root in SEARCH_ROOTS:
        if not Path(root).exists():
            continue

        for p in Path(root).rglob("frozen_subject_split_seed42.json"):
            try:
                h = sha256_file(p)
                candidates.append((p, h))
            except Exception:
                pass

    if not candidates:
        return None, None

    # Accept duplicates only if identical.
    hashes = sorted(set(h for _, h in candidates))

    if len(hashes) > 1:
        print("Warning: multiple non-identical frozen split manifests were found.")
        for p, h in candidates:
            print(h, p)
        return None, None

    return candidates[0][0], candidates[0][1]


split_path, split_file_hash = discover_frozen_split()

if split_path is not None:
    with split_path.open("r") as f:
        frozen_split = json.load(f)

    split_summary = {}

    for split_name in ["train", "val", "test"]:
        tids = frozen_split[split_name]
        subjects = sorted({subject_from_timepoint_id(t) for t in tids})

        split_summary[split_name] = {
            "n_timepoints": len(tids),
            "n_subjects": len(subjects),
        }

    total_timepoints = sum(v["n_timepoints"] for v in split_summary.values())
    total_subjects = sum(v["n_subjects"] for v in split_summary.values())
else:
    frozen_split = None
    split_summary = {
        "train": {"n_timepoints": None, "n_subjects": None},
        "val": {"n_timepoints": None, "n_subjects": None},
        "test": {"n_timepoints": None, "n_subjects": len(ref_subjects)},
    }
    total_timepoints = None
    total_subjects = None

fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 12)
ax.set_ylim(0, 7)
ax.axis("off")

# Boxes are outlines so the figure remains clean in grayscale.
boxes = [
    (0.5, 4.6, 2.3, 1.3, "UCSD-PTGBM\npreprocessed cohort",
     f"{total_subjects if total_subjects is not None else '136'} subjects\n"
     f"{total_timepoints if total_timepoints is not None else '184'} studies"),
    (3.6, 5.1, 2.2, 1.0, "Train",
     f"{split_summary['train']['n_subjects']} subjects / {split_summary['train']['n_timepoints']} studies"
     if split_summary['train']['n_subjects'] is not None else "subject-disjoint"),
    (3.6, 3.7, 2.2, 1.0, "Validation",
     f"{split_summary['val']['n_subjects']} subjects / {split_summary['val']['n_timepoints']} studies"
     if split_summary['val']['n_subjects'] is not None else "subject-disjoint"),
    (3.6, 2.3, 2.2, 1.0, "Development holdout",
     f"{split_summary['test']['n_subjects']} subjects / {split_summary['test']['n_timepoints']} studies"
     if split_summary['test']['n_subjects'] is not None else f"{len(ref_subjects)} subjects"),
]

for x, y, w, h, title, subtitle in boxes:
    ax.add_patch(Rectangle((x, y), w, h, fill=False, linewidth=1.5))
    ax.text(x + w/2, y + h*0.62, title, ha="center", va="center", fontsize=10)
    ax.text(x + w/2, y + h*0.25, subtitle, ha="center", va="center", fontsize=8)

# Main flow arrow.
ax.add_patch(FancyArrowPatch((2.8, 5.25), (3.55, 5.55), arrowstyle="->", mutation_scale=12))
ax.add_patch(FancyArrowPatch((2.8, 5.25), (3.55, 4.2), arrowstyle="->", mutation_scale=12))
ax.add_patch(FancyArrowPatch((2.8, 5.25), (3.55, 2.8), arrowstyle="->", mutation_scale=12))

# Factorial grid.
grid_x = 7.2
grid_y = 2.0
cell_w = 1.9
cell_h = 1.1

ax.text(grid_x + cell_w, grid_y + 3.0, "2 × 2 imaging-input experiment", ha="center", fontsize=11)
ax.text(grid_x + 0.95, grid_y + 2.55, "ADC absent", ha="center", fontsize=9)
ax.text(grid_x + 2.85, grid_y + 2.55, "ADC present", ha="center", fontsize=9)
ax.text(grid_x - 0.15, grid_y + 1.65, "RSI absent", ha="right", va="center", fontsize=9)
ax.text(grid_x - 0.15, grid_y + 0.55, "RSI present", ha="right", va="center", fontsize=9)

grid_labels = [
    (grid_x, grid_y + 1.1, "Baseline\nT1/T1ce/T2/FLAIR"),
    (grid_x + cell_w, grid_y + 1.1, "ADC-only\nBaseline + ADC"),
    (grid_x, grid_y, "RSI-only\nBaseline + RSI"),
    (grid_x + cell_w, grid_y, "Full\nBaseline + RSI + ADC"),
]

for x, y, label in grid_labels:
    ax.add_patch(Rectangle((x, y), cell_w, cell_h, fill=False, linewidth=1.5))
    ax.text(x + cell_w/2, y + cell_h/2, label, ha="center", va="center", fontsize=9)

ax.text(
    grid_x + cell_w,
    grid_y - 0.65,
    "Each arm trained with seeds 42, 43, and 44\n"
    "Primary analysis: seed-averaged subject-level factorial contrasts",
    ha="center",
    va="top",
    fontsize=9,
)

ax.add_patch(FancyArrowPatch((5.85, 2.8), (7.0, 3.1), arrowstyle="->", mutation_scale=12))

ax.set_title("Study design and controlled RSI × ADC factorial analysis", fontsize=13)

fig.tight_layout()
path = save_figure_all_formats(fig, "Fig1_study_design_factorial", main=True)
plt.show()
print("Saved:", path)

## 15. Figure 2 — Paired four-arm subject performance

In [ ]:
arm_order = ["baseline", "adc_only", "rsi_only", "full_6ch"]
x = np.arange(1, len(arm_order) + 1)

fig, ax = plt.subplots(figsize=(8, 5.5))

# Subject-paired trajectories.
for subject_id, row in global_arm_scores[arm_order].iterrows():
    y = row.to_numpy(dtype=float)
    if np.isfinite(y).all():
        ax.plot(x, y, linewidth=0.7, alpha=0.25)

# Distribution summary.
box_data = [global_arm_scores[arm].dropna().to_numpy() for arm in arm_order]
ax.boxplot(box_data, positions=x, widths=0.45, showfliers=False)

# Show individual values without random jitter so the figure is deterministic.
for pos, arm in zip(x, arm_order):
    vals = global_arm_scores[arm].dropna().to_numpy()
    ax.scatter(np.full(len(vals), pos), vals, s=18, alpha=0.65)

ax.set_xticks(x)
ax.set_xticklabels([ARM_DISPLAY[a] for a in arm_order])
ax.set_ylabel("Seed-averaged subject foreground Dice")
ax.set_xlabel("Imaging-input arm")
ax.set_title("Paired subject-level performance across the four imaging arms")

fig.tight_layout()
path = save_figure_all_formats(fig, "Fig2_four_arm_paired_subject_performance", main=True)
plt.show()
print("Saved:", path)

## 16. Figure 3 — Primary RSI-effect forest plot

In [ ]:
forest_rows = [{
    "label": "Overall foreground",
    "effect": float(primary_df.loc[0, "mean_effect"]),
    "lo": float(primary_df.loc[0, "mean_effect_ci95_lo"]),
    "hi": float(primary_df.loc[0, "mean_effect_ci95_hi"]),
}]

for _, row in class_rsi_df.sort_values("class_id").iterrows():
    forest_rows.append({
        "label": row["class_name"],
        "effect": float(row["mean_effect"]),
        "lo": float(row["mean_effect_ci95_lo"]),
        "hi": float(row["mean_effect_ci95_hi"]),
    })

forest_df = pd.DataFrame(forest_rows)
y = np.arange(len(forest_df))[::-1]

fig, ax = plt.subplots(figsize=(7.5, 4.8))

ax.errorbar(
    forest_df["effect"],
    y,
    xerr=[
        forest_df["effect"] - forest_df["lo"],
        forest_df["hi"] - forest_df["effect"],
    ],
    fmt="o",
    capsize=4,
)

ax.axvline(0, linestyle="--", linewidth=1)
ax.set_yticks(y)
ax.set_yticklabels(forest_df["label"])
ax.set_xlabel("RSI factorial main effect on Dice")
ax.set_title("Incremental RSI contribution: overall and tissue-specific effects")

fig.tight_layout()
path = save_figure_all_formats(fig, "Fig3_rsi_effect_forest", main=True)
plt.show()
print("Saved:", path)

## 17. Figure 4 — RSI × ADC factorial interaction plot

In [ ]:
# Extract the two RSI simple effects already computed in Cell 11 (simple_df)
rsi_simple = simple_df[
    simple_df["estimand"].isin([
        "RSI effect when ADC absent",
        "RSI effect when ADC present",
    ])
].copy()

x = np.array([0, 1])
means_simple = rsi_simple["mean_effect"].to_numpy(dtype=float)
ci_lo = rsi_simple["mean_effect_ci95_lo"].to_numpy(dtype=float)
ci_hi = rsi_simple["mean_effect_ci95_hi"].to_numpy(dtype=float)
yerr = np.vstack([means_simple - ci_lo, ci_hi - means_simple])

fig, ax = plt.subplots(figsize=(7.2, 5.2))

# Reference line at 0 (no RSI benefit)
ax.axhline(0, linestyle="--", linewidth=1, color="gray", alpha=0.8)

# Plot paired subject trajectories for transparency (optional subtle background)
for _, row in global_contrasts[["rsi_effect_adc_absent", "rsi_effect_adc_present"]].iterrows():
    y_subj = row.to_numpy(dtype=float)
    if np.isfinite(y_subj).all():
        ax.plot(x, y_subj, linewidth=0.6, alpha=0.18, color="gray")

# Plot mean simple effects + 95% bootstrap CIs
ax.errorbar(
    x,
    means_simple,
    yerr=yerr,
    fmt="o-",
    linewidth=2,
    markersize=7,
    capsize=5,
    label="Mean paired RSI simple effect (±95% bootstrap CI)",
)

for xi, m in zip(x, means_simple):
    ax.annotate(
        f"{m:+.4f}",
        (xi, m),
        textcoords="offset points",
        xytext=(12, 4),
        fontsize=9,
    )

interaction_row = secondary_df.loc[
    secondary_df["estimand"] == "RSI × ADC interaction"
].iloc[0]
interaction_val = float(interaction_row["mean_effect"])
interaction_lo = float(interaction_row["mean_effect_ci95_lo"])
interaction_hi = float(interaction_row["mean_effect_ci95_hi"])
interaction_p = float(interaction_row["wilcoxon_p"])

ax.set_xticks(x)
ax.set_xticklabels([
    "ADC absent\n(RSI-only − Baseline)",
    "ADC present\n(Full 6ch − ADC-only)",
])
ax.set_xlim(-0.35, 1.35)
ax.set_ylabel("Paired RSI simple effect on foreground Dice (Δ Dice)")
ax.set_xlabel("ADC availability")
ax.set_title("Paired RSI simple effects and RSI × ADC factorial interaction")
ax.legend(loc="upper right", frameon=True)

ax.text(
    0.5,
    0.04,
    f"Interaction I(RSI×ADC) = {interaction_val:+.4f} "
    f"(95% CI: {interaction_lo:+.4f} to {interaction_hi:+.4f}, Wilcoxon p = {interaction_p:.3f})",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", fill=False, linewidth=0.8),
)

fig.tight_layout()
path = save_figure_all_formats(fig, "Fig4_rsi_adc_interaction", main=True)
plt.show()
print("Saved:", path)

## 18. Figure 5 — Seed robustness of factorial effects

In [ ]:
seed_plot = seed_robustness_df[
    seed_robustness_df["estimand"].isin([
        "RSI main effect",
        "ADC main effect",
        "RSI × ADC interaction",
    ])
].copy()

fig, ax = plt.subplots(figsize=(8, 5.2))

for estimand in ["RSI main effect", "ADC main effect", "RSI × ADC interaction"]:
    grp = seed_plot[seed_plot["estimand"] == estimand].sort_values("seed")

    y = grp["mean_effect"].to_numpy()
    lo = grp["mean_effect_ci95_lo"].to_numpy()
    hi = grp["mean_effect_ci95_hi"].to_numpy()

    ax.errorbar(
        grp["seed"].to_numpy(),
        y,
        yerr=[y - lo, hi - y],
        marker="o",
        capsize=4,
        label=estimand,
    )

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xticks(SEEDS)
ax.set_xlabel("Training seed")
ax.set_ylabel("Mean subject-level Dice effect")
ax.set_title("Robustness of factorial effects across model initializations")
ax.legend()

fig.tight_layout()
path = save_figure_all_formats(fig, "Fig5_seed_robustness_factorial_effects", main=True)
plt.show()
print("Saved:", path)

## 19. Supplementary Figures S1–S4 — Four-arm performance by tissue compartment

In [ ]:
supp_class_paths = []

for class_id in range(1, 5):
    scores = class_arm_scores[class_id]
    arm_order = ["baseline", "adc_only", "rsi_only", "full_6ch"]
    x = np.arange(1, 5)

    fig, ax = plt.subplots(figsize=(7.5, 5.0))

    # Paired subject trajectories only for rows evaluable in all four arms.
    for subject_id, row in scores[arm_order].iterrows():
        y = row.to_numpy(dtype=float)
        if np.isfinite(y).all():
            ax.plot(x, y, linewidth=0.7, alpha=0.25)

    box_data = [scores[arm].dropna().to_numpy() for arm in arm_order]
    ax.boxplot(box_data, positions=x, widths=0.45, showfliers=False)

    for pos, arm in zip(x, arm_order):
        vals = scores[arm].dropna().to_numpy()
        ax.scatter(np.full(len(vals), pos), vals, s=18, alpha=0.65)

    ax.set_xticks(x)
    ax.set_xticklabels([ARM_DISPLAY[a] for a in arm_order])
    ax.set_ylabel(f"{LABEL_NAMES[class_id]} Dice")
    ax.set_xlabel("Imaging-input arm")
    ax.set_title(
        f"{LABEL_NAMES[class_id]} — {LABEL_LONG_NAMES[class_id]}\n"
        "paired seed-averaged subject performance"
    )

    fig.tight_layout()

    stem = f"FigS{class_id}_{LABEL_NAMES[class_id]}_four_arm_paired_performance"
    path = save_figure_all_formats(fig, stem, main=False)
    supp_class_paths.append(path)
    plt.show()

    print("Saved:", path)

## 20. Supplementary Figure S5 — Subject-level distribution of the RSI effect

In [ ]:
rsi_effects = global_contrasts["rsi_main_effect"].dropna().sort_values()

fig, ax = plt.subplots(figsize=(8, 5.0))

positions = np.arange(1, len(rsi_effects) + 1)
ax.bar(positions, rsi_effects.to_numpy())
ax.axhline(0, linewidth=1)

ax.set_xlabel("Subjects ranked by RSI factorial effect")
ax.set_ylabel("RSI factorial main effect on Dice")
ax.set_title("Heterogeneity of the RSI effect across subjects")
ax.set_xticks([])

fig.tight_layout()
path = save_figure_all_formats(fig, "FigS5_subject_level_rsi_effect_waterfall", main=False)
plt.show()
print("Saved:", path)

## 21. Supplementary Figure S6 — Class evaluability on the development holdout

In [ ]:
evaluable_rows = []

for class_id in range(1, 5):
    metric = f"dice_class_{class_id}"
    scores = class_arm_scores[class_id]

    # Number of subjects with at least one finite seed-averaged class score in each arm.
    for arm in ARMS:
        evaluable_rows.append({
            "class_id": class_id,
            "class_name": LABEL_NAMES[class_id],
            "arm": arm,
            "n_evaluable": int(scores[arm].notna().sum()),
        })

evaluable_df = pd.DataFrame(evaluable_rows)
evaluable_df.to_csv(TABLE_DIR / "Table_class_evaluability.csv", index=False)

# Use the minimum across arms for a conservative single bar per class.
evaluable_min = (
    evaluable_df.groupby(["class_id", "class_name"])["n_evaluable"]
    .min()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(7.2, 4.8))

ax.bar(evaluable_min["class_name"], evaluable_min["n_evaluable"])
ax.set_ylabel("Subjects evaluable in all arms (minimum count)")
ax.set_xlabel("Tissue compartment")
ax.set_title("Class-specific evaluability under the explicit absent-class rule")

fig.tight_layout()
path = save_figure_all_formats(fig, "FigS6_class_evaluability", main=False)
plt.show()
print("Saved:", path)

## 22. Supplementary validation-training trajectories

The 12 run folders contain `training_history.csv`. This cell searches for them and tries to identify an epoch column and a validation-Dice column automatically.

It generates one supplementary figure per arm, with one line for each training seed.

If the column names differ from the expected patterns, the notebook prints the available columns instead of silently plotting the wrong quantity.

In [ ]:
def find_training_history(metric_path):
    candidates = []

    # Run folder and nearby parent folders.
    candidates.append(metric_path.parent / "training_history.csv")

    for parent in list(metric_path.parents)[:5]:
        candidates.append(parent / "training_history.csv")

    for p in candidates:
        if p.exists():
            return p

    return None


def pick_column(columns, exact_candidates=None, contains_all=None):
    columns = list(columns)

    if exact_candidates:
        for c in exact_candidates:
            if c in columns:
                return c

    if contains_all:
        for c in columns:
            name = c.lower()
            if all(token in name for token in contains_all):
                return c

    return None


history = {}
history_inventory = []

for key, metric_path in sorted(registry.items()):
    hp = find_training_history(metric_path)

    if hp is None:
        history_inventory.append({
            "arm": key[0],
            "seed": key[1],
            "status": "MISSING",
            "path": None,
        })
        continue

    df = pd.read_csv(hp)

    epoch_col = pick_column(
        df.columns,
        exact_candidates=["epoch", "Epoch"],
        contains_all=["epoch"],
    )

    val_col = pick_column(
        df.columns,
        exact_candidates=[
            "subject_mean_foreground_dice",      # <-- Matches v6 training notebook
            "subject_median_foreground_dice",    # <-- Fallback from v6 training notebook
            "val_subject_mean_dice",
            "validation_subject_mean_dice",
            "val_foreground_mean_dice",
            "validation_foreground_mean_dice",
            "val_mean_dice",
        ],
    )

    if val_col is None:
        # Fallback: any numeric column containing both ("val" or "subject") and "dice".
        for c in df.columns:
            lc = c.lower()
            if ("val" in lc or "subject" in lc) and "dice" in lc and pd.api.types.is_numeric_dtype(df[c]):
                val_col = c
                break

    status = "OK" if epoch_col is not None and val_col is not None else "UNRESOLVED_COLUMNS"

    history_inventory.append({
        "arm": key[0],
        "seed": key[1],
        "status": status,
        "path": str(hp),
        "epoch_col": epoch_col,
        "val_col": val_col,
        "columns": list(df.columns),
    })

    if status == "OK":
        history[key] = (df, epoch_col, val_col)

history_inventory_df = pd.DataFrame(history_inventory)
display(history_inventory_df)

history_inventory_df.to_json(
    DATA_DIR / "training_history_inventory.json",
    orient="records",
    indent=2,
)

supp_index_start = 7

for offset, arm in enumerate(ARMS):
    arm_histories = [
        (seed, history[(arm, seed)])
        for seed in SEEDS
        if (arm, seed) in history
    ]

    if not arm_histories:
        print(f"{arm}: no usable training history found; no curve generated.")
        continue

    fig, ax = plt.subplots(figsize=(7.5, 4.8))

    for seed, (df, epoch_col, val_col) in arm_histories:
        ax.plot(
            df[epoch_col].to_numpy(),
            df[val_col].to_numpy(),
            label=f"Seed {seed}",
        )

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Validation subject-level foreground Dice")
    ax.set_title(f"{ARM_DISPLAY[arm]} validation trajectory")
    ax.legend()

    fig.tight_layout()

    fig_num = supp_index_start + offset
    stem = f"FigS{fig_num}_{arm}_validation_trajectory"
    path = save_figure_all_formats(fig, stem, main=False)
    plt.show()

    print("Saved:", path)

## 23. Qualitative segmentation figure availability check

The statistical CSVs contain performance measurements, **not voxelwise prediction masks**.

A manuscript-quality qualitative figure comparing:

`T1ce | FLAIR | RSI | ADC | Ground truth | Baseline | ADC-only | RSI-only | Full`

requires either:

1. saved prediction masks from the completed runs, or
2. a later inference-only pass on a few statistically selected cases using the frozen best checkpoints.

No retraining is required.

This cell checks whether any obvious prediction-mask files were included in the 12-run dataset.

In [ ]:
import gc
import subprocess
import sys
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Ensure MONAI and einops are available in the analysis environment
try:
    import torch
    import einops
    from monai.networks.nets import SwinUNETR
    from monai.inferers import sliding_window_inference
except ImportError:
    print("Installing MONAI and einops for checkpoint inference...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "monai==1.6.0", "einops"])
    import torch
    import einops
    from monai.networks.nets import SwinUNETR
    from monai.inferers import sliding_window_inference

# ---------------------------------------------------------
# 1. Select 3 representative subjects from global contrasts
# ---------------------------------------------------------
rsi_series = global_contrasts["rsi_main_effect"].dropna().sort_values()

neg_subj = str(rsi_series.index[0])
med_subj = str((rsi_series - rsi_series.median()).abs().idxmin())
pos_subj = str(rsi_series.index[-1])

SELECTED_SUBJECTS = [
    ("High RSI benefit", pos_subj, float(rsi_series.loc[pos_subj])),
    ("Near-neutral (median)", med_subj, float(rsi_series.loc[med_subj])),
    ("Negative RSI effect", neg_subj, float(rsi_series.loc[neg_subj])),
]

print("Statistically selected qualitative subjects:")
for label, sid, eff in SELECTED_SUBJECTS:
    print(f"  {label:24s}: {sid} (Δ_RSI = {eff:+.4f})")

# ---------------------------------------------------------
# 2. Discover 12 best_checkpoint.pt files & verify SHA256
# ---------------------------------------------------------
hash_to_run = {}
for key, obj in metadata.items():
    if obj and "run_config_sha256" in obj:
        hash_to_run[obj["run_config_sha256"]] = key

ckpt_registry = {}
ARM_RE = re.compile(r"(baseline|adc_only|rsi_only|full_6ch)")
SEED_RE = re.compile(r"seed[\s_-]*(42|43|44)", re.IGNORECASE)

for root in SEARCH_ROOTS:
    if not Path(root).exists():
        continue
    for p in Path(root).rglob("best_checkpoint.pt"):
        try:
            ckpt_meta = torch.load(p, map_location="cpu", weights_only=False)
        except Exception as exc:
            print(f"Skipping unreadable checkpoint {p}: {exc}")
            continue

        rc_hash = ckpt_meta.get("run_config_sha256")
        sp_hash = ckpt_meta.get("split_sha256")

        if sp_hash != ref_config.get("split_sha256"):
            continue

        if rc_hash in hash_to_run:
            run_key = hash_to_run[rc_hash]
        else:
            p_str = str(p)
            m_arm = ARM_RE.search(p_str)
            m_seed = SEED_RE.search(p_str)
            if m_arm and m_seed:
                run_key = (m_arm.group(1), int(m_seed.group(1)))
            else:
                continue

        ckpt_registry[run_key] = p
        del ckpt_meta

print(f"\nDiscovered verified best_checkpoint.pt files: {len(ckpt_registry)} / 12")
for k in sorted(ckpt_registry):
    print(f"  {k}: {ckpt_registry[k]}")

# ---------------------------------------------------------
# 3. Locate preprocessed 6-channel NPZ volumes for the 3 cases
# ---------------------------------------------------------
def find_rsi_npz_for_subject(subject_id):
    candidates = []
    for root in SEARCH_ROOTS:
        if not Path(root).exists():
            continue
        for p in Path(root).rglob(f"{subject_id}_*.npz"):
            if "rsi" in str(p.parent).lower():
                return p
            candidates.append(p)
    return sorted(candidates)[0] if candidates else None

subject_npz_paths = {}
for _, sid, _ in SELECTED_SUBJECTS:
    npz_p = find_rsi_npz_for_subject(sid)
    if npz_p is not None:
        subject_npz_paths[sid] = npz_p

print(f"\nLocated preprocessed NPZ volumes for selected subjects: {len(subject_npz_paths)} / 3")
for sid, p in subject_npz_paths.items():
    print(f"  {sid}: {p}")

# ---------------------------------------------------------
# 4. Run inference & build 9-panel qualitative figure
# ---------------------------------------------------------
PRED_DIR = DATA_DIR / "qualitative_predictions"
PRED_DIR.mkdir(parents=True, exist_ok=True)

if len(ckpt_registry) == 12 and len(subject_npz_paths) == 3:
    infer_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\nRunning inference on device: {infer_device}")

    ROI_SIZE = tuple(ref_config.get("roi_size", [96, 96, 96]))
    FEATURE_SIZE = int(ref_config.get("feature_size", 48))
    NUM_CLASSES = int(ref_config.get("num_classes", 5))
    SW_OVERLAP = float(ref_config.get("sliding_window_overlap", 0.5))
    SW_MODE = str(ref_config.get("sliding_window_mode", "gaussian"))

    raw_cases = {}
    for _, sid, _ in SELECTED_SUBJECTS:
        with np.load(subject_npz_paths[sid], allow_pickle=False) as d:
            img = np.asarray(d["image"], dtype=np.float32)
            lbl = np.asarray(d["label"], dtype=np.int64)[0]
        raw_cases[sid] = {"image": img, "label": lbl, "timepoint_id": subject_npz_paths[sid].stem}

    prob_accum = {
        sid: {
            arm: np.zeros((NUM_CLASSES,) + raw_cases[sid]["label"].shape, dtype=np.float32)
            for arm in ARMS
        }
        for _, sid, _ in SELECTED_SUBJECTS
    }

    for (arm, seed), ckpt_path in sorted(ckpt_registry.items()):
        ch_idx = EXPECTED_CHANNEL_INDICES[arm]
        model = SwinUNETR(
            in_channels=len(ch_idx),
            out_channels=NUM_CLASSES,
            feature_size=FEATURE_SIZE,
            use_checkpoint=True,
            spatial_dims=3,
        ).to(infer_device)

        ckpt = torch.load(ckpt_path, map_location=infer_device, weights_only=False)
        model.load_state_dict(ckpt["model_state_dict"], strict=True)
        model.eval()

        with torch.no_grad():
            for _, sid, _ in SELECTED_SUBJECTS:
                img_arm = raw_cases[sid]["image"][ch_idx]
                orig_shape = img_arm.shape[1:]

                pad_width = [(0, 0)] + [
                    (0, max(0, ROI_SIZE[i] - orig_shape[i])) for i in range(3)
                ]
                if any(pw[1] > 0 for pw in pad_width):
                    img_padded = np.pad(img_arm, pad_width, mode="constant", constant_values=0)
                else:
                    img_padded = img_arm

                inp = torch.from_numpy(img_padded[None]).to(infer_device)
                with torch.amp.autocast("cuda", enabled=(infer_device.type == "cuda")):
                    logits = sliding_window_inference(
                        inputs=inp,
                        roi_size=ROI_SIZE,
                        sw_batch_size=1,
                        predictor=model,
                        overlap=SW_OVERLAP,
                        mode=SW_MODE,
                    )
                    probs = torch.softmax(logits, dim=1)[0].detach().cpu().numpy()

                probs = probs[:, :orig_shape[0], :orig_shape[1], :orig_shape[2]]
                prob_accum[sid][arm] += probs / float(len(SEEDS))

        print(f"  Completed inference: {arm}_seed{seed}")
        del model, ckpt
        gc.collect()
        if infer_device.type == "cuda":
            torch.cuda.empty_cache()

    # Save compact 3D segmentation masks (keeps ZIP size small)
    pred_masks = {sid: {} for _, sid, _ in SELECTED_SUBJECTS}
    for _, sid, _ in SELECTED_SUBJECTS:
        save_dict = {
            "gt_label": raw_cases[sid]["label"].astype(np.uint8),
        }
        for arm in ARMS:
            mask = np.argmax(prob_accum[sid][arm], axis=0).astype(np.uint8)
            pred_masks[sid][arm] = mask
            save_dict[f"pred_{arm}"] = mask
        np.savez_compressed(PRED_DIR / f"{raw_cases[sid]['timepoint_id']}_predictions.npz", **save_dict)

    # 9-Column Qualitative Figure:
    # T1ce | FLAIR | RSI | ADC | Ground truth | Baseline | ADC-only | RSI-only | Full
    seg_cmap = ListedColormap(["none", "#e41a1c", "#4daf4a", "#ff7f00", "#377eb8"])
    col_titles = [
        "T1ce", "FLAIR", "RSI", "ADC",
        "Ground Truth", "Baseline", "ADC-only", "RSI-only", "Full (6-ch)"
    ]

    fig, axes = plt.subplots(3, 9, figsize=(18, 6.6))

    for row_idx, (case_label, sid, eff) in enumerate(SELECTED_SUBJECTS):
        img6 = raw_cases[sid]["image"]
        gt = raw_cases[sid]["label"]

        fg_counts = (gt > 0).sum(axis=(0, 1))
        z = int(np.argmax(fg_counts)) if fg_counts.max() > 0 else gt.shape[2] // 2

        t1ce_slice = np.rot90(img6[1, :, :, z])
        flair_slice = np.rot90(img6[3, :, :, z])
        rsi_slice = np.rot90(img6[4, :, :, z])
        adc_slice = np.rot90(img6[5, :, :, z])
        gt_slice = np.rot90(gt[:, :, z])

        modalities = [t1ce_slice, flair_slice, rsi_slice, adc_slice]
        for c_idx, mod_slice in enumerate(modalities):
            ax = axes[row_idx, c_idx]
            vmin, vmax = np.percentile(mod_slice, [2, 98])
            ax.imshow(mod_slice, cmap="gray", vmin=vmin, vmax=vmax)
            ax.axis("off")
            if row_idx == 0:
                ax.set_title(col_titles[c_idx], fontsize=10, fontweight="bold")

        overlay_masks = [gt_slice] + [np.rot90(pred_masks[sid][arm][:, :, z]) for arm in ARMS]
        vmin_bg, vmax_bg = np.percentile(t1ce_slice, [2, 98])

        for m_offset, mask_slice in enumerate(overlay_masks):
            c_idx = 4 + m_offset
            ax = axes[row_idx, c_idx]
            ax.imshow(t1ce_slice, cmap="gray", vmin=vmin_bg, vmax=vmax_bg)
            masked_seg = np.ma.masked_where(mask_slice == 0, mask_slice)
            ax.imshow(masked_seg, cmap=seg_cmap, vmin=0, vmax=4, alpha=0.65, interpolation="nearest")
            ax.axis("off")
            if row_idx == 0:
                ax.set_title(col_titles[c_idx], fontsize=10, fontweight="bold")

        axes[row_idx, 0].text(
            -0.08, 0.5,
            f"{case_label}\n{sid}\n(Δ_RSI = {eff:+.3f})",
            transform=axes[row_idx, 0].transAxes,
            va="center", ha="right", fontsize=9, fontweight="bold",
        )

    legend_patches = [
        Patch(facecolor="#e41a1c", label="NETC (Class 1)"),
        Patch(facecolor="#4daf4a", label="SNFH (Class 2)"),
        Patch(facecolor="#ff7f00", label="ET (Class 3)"),
        Patch(facecolor="#377eb8", label="RC (Class 4)"),
    ]
    fig.legend(handles=legend_patches, loc="lower center", ncol=4, frameon=False, fontsize=9.5)
    fig.suptitle(
        "Qualitative segmentation comparisons across high-benefit, near-neutral, and negative RSI-effect cases",
        fontsize=12, fontweight="bold", y=0.98
    )
    fig.tight_layout(rect=[0.08, 0.05, 1.0, 0.94])

    qual_fig_path = save_figure_all_formats(fig, "Fig6_qualitative_segmentation_examples", main=True)
    plt.show()
    print("Saved qualitative figure:", qual_fig_path)

    qualitative_status = {
        "status": "GENERATED_FROM_FROZEN_BEST_CHECKPOINTS",
        "selected_subjects": [
            {"category": cat, "subject_id": sid, "rsi_main_effect": eff}
            for cat, sid, eff in SELECTED_SUBJECTS
        ],
        "figure_path": str(qual_fig_path),
    }
else:
    print(
        "\nSkipping live checkpoint inference (need all 12 best_checkpoint.pt files "
        "and the preprocessed_rsi_5class .npz dataset attached)."
    )
    qualitative_status = {
        "status": "PENDING_CHECKPOINT_AND_NPZ_ATTACHMENT",
        "checkpoints_found": len(ckpt_registry),
        "subject_npz_found": len(subject_npz_paths),
        "selected_subjects": [
            {"category": cat, "subject_id": sid, "rsi_main_effect": eff}
            for cat, sid, eff in SELECTED_SUBJECTS
        ],
    }

with (OUTPUT_DIR / "qualitative_figure_status.json").open("w") as f:
    json.dump(qualitative_status, f, indent=2)

## 24. Manuscript-oriented summary tables

In [ ]:
# Compact global factorial table.
global_factorial_table = pd.concat(
    [
        primary_df.assign(order=1),
        secondary_df.assign(order=[2, 3]),
    ],
    ignore_index=True,
    sort=False,
).sort_values("order")

global_factorial_table.to_csv(
    TABLE_DIR / "Manuscript_Table_global_factorial_effects.csv",
    index=False,
)

# Compact class RSI table.
manuscript_class_table = class_rsi_df[
    [
        "class_id",
        "class_name",
        "class_long_name",
        "n_subjects",
        "mean_effect",
        "mean_effect_ci95_lo",
        "mean_effect_ci95_hi",
        "median_effect",
        "wilcoxon_p",
        "holm_p",
        "rank_biserial",
        "positive_fraction",
    ]
].copy()

manuscript_class_table.to_csv(
    TABLE_DIR / "Manuscript_Table_class_specific_rsi_effects.csv",
    index=False,
)

print("Global factorial table:")
display(global_factorial_table)

print("\nClass-specific RSI table:")
display(manuscript_class_table)

## 25. Analysis manifest, results summary JSON, and final ZIP

In [ ]:
manifest = {
    "analysis_name": "UCSD_PTGBM_Final_Factorial_Statistics_and_Figures",
    "analysis_status": ANALYSIS_STATUS,
    "primary_split": PRIMARY_SPLIT,
    "arms": ARMS,
    "arm_display": ARM_DISPLAY,
    "seeds": SEEDS,
    "expected_channel_indices": EXPECTED_CHANNEL_INDICES,
    "channel_names": {str(k): v for k, v in CHANNEL_NAMES.items()},
    "label_mapping": {str(k): v for k, v in LABEL_NAMES.items()},
    "label_long_names": {str(k): v for k, v in LABEL_LONG_NAMES.items()},
    "primary_estimand": "RSI main effect",
    "key_secondary_estimands": [
        "ADC main effect",
        "RSI × ADC interaction",
    ],
    "bootstrap_resamples": N_BOOT,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "provenance": {
        "split_sha256": ref_config.get("split_sha256"),
        "dataset_id_sha256": ref_config.get("dataset_id_sha256"),
        "label_content_sha256": ref_config.get("label_content_sha256"),
        "structural_content_sha256": ref_config.get("structural_content_sha256"),
        "advanced_channel_content_sha256": ref_config.get("advanced_channel_content_sha256"),
        "pretrained_sha256": ref_config.get("pretrained_sha256"),
    },
    "run_files": [
        {
            "arm": arm,
            "seed": seed,
            "subject_metrics_path": str(path),
            "subject_metrics_sha256": sha256_file(path),
            "run_metadata_path": str(find_nearby_metadata(path)),
        }
        for (arm, seed), path in sorted(registry.items())
    ],
}

with (OUTPUT_DIR / "analysis_manifest.json").open("w") as f:
    json.dump(manifest, f, indent=2)


def row_to_clean_dict(row):
    out = {}
    for key, value in row.items():
        if isinstance(value, (np.integer,)):
            out[key] = int(value)
        elif isinstance(value, (np.floating,)):
            out[key] = None if not np.isfinite(value) else float(value)
        elif isinstance(value, (bool, np.bool_)):
            out[key] = bool(value)
        else:
            out[key] = value
    return out


results_summary = {
    "primary": row_to_clean_dict(primary_df.iloc[0].to_dict()),
    "key_secondary": [
        row_to_clean_dict(r)
        for r in secondary_df.to_dict(orient="records")
    ],
    "class_specific_rsi": [
        row_to_clean_dict(r)
        for r in class_rsi_df.to_dict(orient="records")
    ],
    "simple_effects": [
        row_to_clean_dict(r)
        for r in simple_df.to_dict(orient="records")
    ],
    "arm_descriptive": [
        row_to_clean_dict(r)
        for r in arm_descriptive_df.to_dict(orient="records")
    ],
}

with (OUTPUT_DIR / "paper_results_summary.json").open("w") as f:
    json.dump(results_summary, f, indent=2)


readme = """
UCSD-PTGBM FINAL FACTORIAL ANALYSIS OUTPUTS

MAIN FIGURES
Fig1: Study design and 2x2 RSI x ADC factorial framework
Fig2: Paired seed-averaged subject performance across four arms
Fig3: Overall and tissue-specific RSI-effect forest plot
Fig4: RSI x ADC interaction plot
Fig5: Seed robustness of factorial effects

SUPPLEMENTARY FIGURES
FigS1-S4: Per-class four-arm paired performance
FigS5: Subject-level RSI-effect waterfall
FigS6: Class evaluability
FigS7-S10: Per-arm validation trajectories, when training-history columns are resolvable

TABLES
Primary RSI effect, key secondary global effects, class-specific RSI effects,
exploratory class effects/interactions, simple effects, arm descriptives,
seed robustness, and manuscript-oriented compact tables.

IMPORTANT
The internal_test split is a development holdout because it has already been inspected.
A qualitative MRI/segmentation figure cannot be produced from summary CSVs alone unless
voxelwise prediction masks are available. It can be generated later with inference only
from frozen checkpoints, without retraining.
""".strip()

with (OUTPUT_DIR / "README.txt").open("w") as f:
    f.write(readme)

zip_path = shutil.make_archive(
    str(OUTPUT_DIR),
    "zip",
    root_dir=OUTPUT_DIR,
)

print("Analysis manifest:", OUTPUT_DIR / "analysis_manifest.json")
print("Paper results summary:", OUTPUT_DIR / "paper_results_summary.json")
print("README:", OUTPUT_DIR / "README.txt")
print("Final ZIP:", zip_path)

print("\nMain figures:")
for p in sorted(FIG_MAIN.glob("*.png")):
    print(" ", p.name)

print("\nSupplementary figures:")
for p in sorted(FIG_SUPP.glob("*.png")):
    print(" ", p.name)

print("\nTables:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print(" ", p.name)

# Final interpretation rule

When this notebook finishes, treat the generated output folder as the **single statistical source of truth** for manuscript writing.

The Results section should follow this hierarchy:

1. **Primary:** RSI factorial main effect on foreground mean Dice.
2. **Key secondary:** ADC main effect and RSI × ADC interaction.
3. **Mechanistic secondary:** NETC, SNFH, ET, and RC-specific RSI effects.
4. **Simple effects:** interpret any interaction.
5. **Seed robustness:** demonstrate stability across seeds 42/43/44.
6. **Descriptive arm performance:** context only.
7. **Qualitative examples:** generated later from frozen inference if prediction masks are not already available.

Do not promote `Full − Baseline` above the factorial RSI estimand simply because it is visually intuitive.